# 01 · Data overview

What the hand-to-nose web app has recorded so far, where it is stored and what it looks like.

**How to run**

1. Install the packages once: `pip install -r analysis/requirements.txt`
2. Put the Supabase secret key into a file `.env` in the repository root:
   `SUPABASE_SECRET_KEY=sb_secret_...` (Supabase → Project Settings → API Keys).
   `.env` is ignored by git.
3. Download the data: `python analysis/export_data.py` → `data/raw/` (also ignored by git).
4. Run all cells: *Kernel → Restart Kernel and Run All Cells*. Run the export again whenever new
   participants have done the task.

**Privacy.** This notebook is committed with its outputs so it can be read on GitHub. It therefore
shows the participants only in aggregate (counts, age range); it never prints age or sex per
person. Sessions are identified only by their random code.

## 1. Where the data live

```
Supabase project "hand-to-nose"  (database in Frankfurt)
├── sessions   one row per run of the app = one person
│              id, age, sex, edinburgh_lq (handedness), device (phone, browser), is_pilot, ...
└── trials     one row per successful movement (to the nose and back), linked by session_id
               hand, speed, times of the events (high beep, nose touch, nose off, still again)
               samples ── the RAW SENSOR DATA of this movement, ~60 values per second:
                          t (ms), ax ay az, lx ly lz, gx gy gz, oa ob og
```

The raw data are not separate columns: each movement keeps them in **one JSON field,
`trials.samples`**, which holds one array per sensor channel. In the Supabase table editor this
looks like a single cell of text. `analysis/export_data.py` copies both tables into `data/raw/`,
and `analysis/loading.py` unpacks them into pandas tables.

| Key | What it is | Unit |
|---|---|---|
| `t` | time from the start of the recording | ms |
| `ax ay az` | acceleration **with** gravity (at rest `ay` ≈ +9.8 when the phone is upright) | m/s² |
| `lx ly lz` | acceleration **without** gravity (the movement itself) | m/s² |
| `gx gy gz` | rotation rate about the phone's x, y, z axes | deg/s |
| `oa ob og` | orientation angles alpha, beta, gamma | deg |

Phone axes (portrait, screen facing the person): **x** to the right, **y** up, **z** out of the
screen towards the face. Full description: [docs/data_format.md](../docs/data_format.md).

In [ ]:
import sys
from pathlib import Path

# make analysis/ importable, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "analysis" / "loading.py").exists())
sys.path.insert(0, str(ROOT / "analysis"))

import matplotlib.pyplot as plt
import pandas as pd

from loading import (export_info, load_raw, participant_summary, samples_table,
                     sessions_table, trials_table)
from plots import apply_style, plot_reach_profiles, plot_reach_times, plot_trial

apply_style()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

In [ ]:
raw_sessions, raw_trials = load_raw()       # the files exactly as exported

sessions = sessions_table(raw_sessions)    # one row per session
trials = trials_table(raw_trials, sessions)  # one row per movement
samples = samples_table(raw_trials, trials)  # one row per sensor sample

print("Export made:", export_info().get("exported_at", "?"))
print(f"{len(sessions)} sessions, {len(trials)} trials, {len(samples):,} sensor samples")

### What one raw trial looks like

One row of the `trials` table, and the first samples inside its `samples` field:

In [ ]:
example = raw_trials[0]
print("Fields of one trial row:", ", ".join(example))
print("Arrays inside 'samples':", ", ".join(example["samples"]))
pd.DataFrame({k: v[:6] for k, v in example["samples"].items()})

## 2. Sessions

A **pilot** session is a test run (by the researcher or during development). Pilots stay in the
database but are excluded from the analysis. `test` sessions come from the test mode
(final exam), not from data collection.

In [ ]:
trials_per_session = trials["session_id"].value_counts()
(sessions
 .assign(date=sessions["created_at"].dt.tz_convert("Asia/Jerusalem").dt.strftime("%d %b %H:%M"),
         trials=sessions["session_id"].map(trials_per_session).fillna(0).astype(int))
 [["code", "date", "mode", "is_pilot", "app_version", "platform", "browser", "trials"]])

**Which sessions are analysed:** data-collection sessions that are not pilots. While there
are none yet, the rest of the notebook uses the pilot sessions, so that there is something to
look at. Once real participants exist this switches automatically.

In [ ]:
collect = sessions["mode"] == "collect"
study_ids = sessions.loc[collect & ~sessions["is_pilot"], "session_id"]
USE_PILOTS = len(study_ids) == 0
use_ids = sessions.loc[collect, "session_id"] if USE_PILOTS else study_ids
print("Using", "PILOT sessions (no study data yet)" if USE_PILOTS else "study sessions only",
      f"→ {len(use_ids)} sessions")

use_sessions = sessions[sessions["session_id"].isin(use_ids)]
use_trials = trials[trials["session_id"].isin(use_ids)]
use_samples = samples[samples["session_id"].isin(use_ids)]

participant_summary(use_sessions)

## 3. Trials

One row per movement. Durations are computed from the event times:

- `reach_s` — high beep → nose touch (the movement to the nose)
- `at_nose_s` — nose touch → nose off the screen
- `return_s` — nose off → phone still again (the way back)
- `n_failed_before` — attempts discarded before this one (e.g. moved before the beep)

In [ ]:
cols = ["code", "block_index", "trial_index", "hand", "speed", "reach_s", "at_nose_s",
        "return_s", "n_failed_before", "n_samples", "rate_hz"]
use_trials[cols].head(12).round(2)

In [ ]:
# number of movements and median reach time, per instructed speed and hand
(use_trials
 .pivot_table(index="speed", columns="hand", values="reach_s", aggfunc=["count", "median"])
 .reindex(["fast", "normal", "slow"])
 .round(2))

In [ ]:
fig = plot_reach_times(use_trials)
plt.show()

## 4. Data quality

The app asks the browser for ~60 samples per second. Two things are checked per platform:

- the time between samples (`median_dt_ms` should be ≈ 16.7 ms; `worst_gap_ms` shows pauses);
- **repeated readings** — samples whose 3 acceleration and 3 rotation values are all identical to
  the previous sample. Then the browser delivered the same reading twice, and the real sampling
  rate is lower than the nominal one. On the Samsung pilots 15–18 % of the samples repeated the
  previous acceleration reading and 11–13 % the previous rotation reading; on the iPhones about 0 %.
  The table counts a sample as repeated only if all six values repeat.

In [ ]:
(use_trials
 .groupby("platform")
 .agg(sessions=("session_id", "nunique"), trials=("trial_id", "size"),
      rate_hz=("rate_hz", "median"), median_dt_ms=("median_dt_ms", "median"),
      worst_gap_ms=("max_dt_ms", "max"), repeated_share=("repeated_share", "mean"))
 .assign(repeated=lambda d: (100 * d["repeated_share"]).round(1).astype(str) + " %")
 .drop(columns="repeated_share")
 .round(1))

## 5. One movement in detail

All sensor channels of one movement against time from the high beep. The shaded part is the
reach to the nose; dashed lines are the events. For iPhone sessions the accelerations are
already flipped to the Android sign convention (iPhone reports them with the opposite sign).
Change `n` to look at another movement.

In [ ]:
n = 0
trial = use_trials.iloc[n]
one = use_samples[use_samples["trial_id"] == trial["trial_id"]]
fig = plot_trial(one, trial)
plt.show()

## 6. Right hand vs left hand

Every movement to the nose, with time normalised so that 0 = high beep and 1 = nose touch;
this puts slow and fast movements on the same axis. Thin lines are single movements, thick lines
the mean per hand. **Channels where the two thick lines separate consistently are candidates for
features** of the classifier. Different people are mixed here, so a difference can also come
from who did which hand; the real test is leave-one-subject-out.

In [ ]:
fig = plot_reach_profiles(use_samples, ["ax", "ay", "az", "gx", "gy", "gz"])
plt.show()

In [ ]:
# the same for the acceleration without gravity (the movement itself)
fig = plot_reach_profiles(use_samples, ["lx", "ly", "lz"])
plt.show()

## 7. Notes and next steps

- iPhone: both accelerations (`ax…az`, `lx…lz`) have the opposite sign to Android and are flipped
  in `loading.py`. Checked on the pilots: on iPhone, *with gravity − without gravity* is exactly
  the gravity vector. The sign of the rotation rate (`gx…gz`) on iPhone is **not checked yet**.
- Next notebook: one row of features per movement (e.g. peak rotation about each axis, mean
  sideways acceleration during the reach) → logistic regression, evaluated leave-one-subject-out
  (`sklearn.model_selection.LeaveOneGroupOut` with `groups = session_id`).
- Before committing this notebook with outputs: *Restart Kernel and Run All Cells*, and check that
  no output shows age or sex per person.